In [2]:
### set up the notebook
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.signal

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
np.set_printoptions(linewidth=100) 

plt.rcParams.update({'font.size': 14})

In [3]:
### load the GNSS data for each site

# GNSS
sn40 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b011_sn40_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
sn20 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b008_sn20_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
sn06 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b012_sn06_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
ss05 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b014_ss05_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
ss20 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b015_ss20_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
ss30 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b013_ss30_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
ss40 = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b010_ss40_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
swxt = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b007_swxt_soln_v1_JPLfixed_merged_smooth_hr_small.nc')
sext = xr.open_dataset('/data/hendreya/troposphere/data/GNSS/b009_sext_soln_v1_JPLfixed_merged_smooth_hr_small.nc')


In [4]:
### Reindex all to common time period

new_time = np.arange(np.datetime64('2023-04-01T00:00:00'), np.datetime64('2023-07-01T00:00:00'), np.timedelta64(1, 'h'))

# GNSS
sn40 = sn40.reindex(time=new_time, method='nearest', tolerance='10 min')
sn20 = sn20.reindex(time=new_time, method='nearest', tolerance='10 min')
sn06 = sn06.reindex(time=new_time, method='nearest', tolerance='10 min')
ss05 = ss05.reindex(time=new_time, method='nearest', tolerance='10 min')
ss20 = ss20.reindex(time=new_time, method='nearest', tolerance='10 min')
ss30 = ss30.reindex(time=new_time, method='nearest', tolerance='10 min')
ss40 = ss40.reindex(time=new_time, method='nearest', tolerance='10 min')
swxt = swxt.reindex(time=new_time, method='nearest', tolerance='10 min')
sext = sext.reindex(time=new_time, method='nearest', tolerance='10 min')


In [5]:
### format each to 2d array with time, site, for saving into single xarray dataset

GNSS_combined = np.vstack((sn40.tropest_wet.values, 
                           sn20.tropest_wet.values, 
                           sn06.tropest_wet.values, 
                           ss05.tropest_wet.values, 
                           ss20.tropest_wet.values, 
                           ss30.tropest_wet.values, 
                           ss40.tropest_wet.values, 
                           swxt.tropest_wet.values, 
                           sext.tropest_wet.values))

print(np.shape(GNSS_combined))

(9, 2184)


In [8]:
### Create the xarray Dataset

GNSS_hourly = xr.Dataset(data_vars={'sn40': ('time', sn40.tropest_wet.values),
                                        'sn20': ('time', sn20.tropest_wet.values),
                                        'sn06': ('time', sn06.tropest_wet.values),
                                        'ss05': ('time', ss05.tropest_wet.values),
                                        'ss20': ('time', ss20.tropest_wet.values),
                                        'ss30': ('time', ss30.tropest_wet.values),
                                        'ss40': ('time', ss40.tropest_wet.values),
                                        'swxt': ('time', swxt.tropest_wet.values),
                                        'sext': ('time', sext.tropest_wet.values)},
                             coords={'time': new_time}, 
                             attrs={'info': 'Hourly Zenith Wet Delay (i.e. wet path delay) data from GNSS buoys in meters.'})

In [10]:
### Save dataset

GNSS_hourly.to_netcdf('../data_to_publish/GNSS_hourly.nc')